<a href="https://colab.research.google.com/github/tselane2110/Extractive-Quesion-Answering/blob/main/ExtractiveQuestionAnswering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Downloading Dependencies

In [ ]:
!pip install torch transformers numpy tqdm
!python squad_span_qa.py --approach token_cls --max_train_samples 2000 --max_eval_samples 500 --epochs 1

### Cloning Git Repo

In [2]:
!git clone https://github.com/tselane2110/Extractive-Quesion-Answering
%cd /content/Extractive-Quesion-Answering

Cloning into 'Extractive-Quesion-Answering'...
remote: Enumerating objects: 12, done.
remote: Counting objects: 100% (12/12), done.
remote: Compressing objects: 100% (10/10), done.
remote: Total 12 (delta 2), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (12/12), 11.67 KiB | 5.83 MiB/s, done.
Resolving deltas: 100% (2/2), done.
/content/Extractive-Quesion-Answering


## Model#1: Start/End Token Classification Using BERT-Base, Uncased

In [3]:
!python squad_span_qa.py --approach token_cls                          # 1. BERT + linear start/end heads

config.json: 100% 570/570 [00:00<00:00, 2.52MB/s]
tokenizer_config.json: 100% 48.0/48.0 [00:00<00:00, 243kB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 5.07MB/s]
tokenizer.json: 100% 466k/466k [00:00<00:00, 9.82MB/s]

model.safetensors: downloading bytes:  72% 318M/440M [00:01<00:00, 353MB/s, 26.0MB/s  ]
model.safetensors: downloading bytes:  90% 397M/440M [00:01<00:00, 367MB/s, 34.4MB/s  ]
model.safetensors: downloading bytes: 100% 415M/415M [00:02<00:00, 165MB/s, 38.0MB/s  ]
model.safetensors: reconstructing file: 100% 440M/440M [00:02<00:00, 175MB/s, 41.2MB/s  ]
Loading weights: 100% 199/199 [00:00<00:00, 6671.89it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
c

### Error Analysis

In [14]:
import json
import random
import re
import string
import urllib.request
import pandas as pd

SQUAD_DEV_URL = "https://rajpurkar.github.io/SQuAD-explorer/dataset/dev-v1.1.json"
PREDICTIONS_PATH = "/content/drive/MyDrive/MS-AI/NLP-ProgrammingAssignment/ProgrammingAssignment-2/Extractive-Quesion-Answering/runs/token_cls/predictions.json"
SEED = 42
NUM_SAMPLES = 10


def normalize_answer(s: str) -> str:
    """Official SQuAD v1.1 answer normalization."""
    s = s.lower()
    s = "".join(ch for ch in s if ch not in set(string.punctuation))
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    return " ".join(s.split())


def get_error_df() -> pd.DataFrame:
    with urllib.request.urlopen(SQUAD_DEV_URL) as response:
        squad_data = json.loads(response.read().decode("utf-8"))

    with open(PREDICTIONS_PATH, "r", encoding="utf-8") as f:
        predictions = json.load(f)

    incorrect = []

    for article in squad_data["data"]:
        for paragraph in article["paragraphs"]:
            for qa in paragraph["qas"]:
                qid = qa["id"]
                ground_truths = [ans["text"] for ans in qa["answers"]]
                pred = predictions.get(qid, "")

                if normalize_answer(pred) not in [normalize_answer(gt) for gt in ground_truths]:
                    incorrect.append({
                        "ID": qid,
                        "Question": qa["question"],
                        "Prediction": pred,
                        "Ground Truth": " | ".join(dict.fromkeys(ground_truths)),
                    })

    random.seed(SEED)
    samples = random.sample(incorrect, min(NUM_SAMPLES, len(incorrect)))
    print(f"Total incorrect (Exact Match): {len(incorrect)}")
    return pd.DataFrame(samples)


df = get_error_df()

# Prevent pandas from truncating long questions/answers in terminal output
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 1000)

print(df.to_string(index=False))

Total incorrect (Exact Match): 2089
                      ID                                                                                                              Question                                                       Prediction                                                                                                                                                                                                                                                       Ground Truth
56f85bb8aef2371900626013                          What unintended development did the relationship with the elector have on church government?                                                     questionable                                                                                                                                                                                      under the temporal sovereign | church government under the temporal sovereign
56d9c551dc89441400fdb7d0      

## Model#2: Reinforcement Learning Using BERT-Base, Uncased

In [4]:
!python squad_span_qa.py --approach rl --init_from runs/token_cls/best # 2. policy-gradient fine-tuning

dev features: 100% 6/6 [00:02<00:00,  2.26it/s]
train questions: 83752  val questions: 3847  dev questions: 10570
train features: 100% 42/42 [00:17<00:00,  2.37it/s]
val features: 100% 2/2 [00:00<00:00,  2.40it/s]
train windows: 84665
epoch 1/3 [RL+CE]: 100% 2646/2646 [12:54<00:00,  3.42it/s, ce=0.533, reward=0.829]
Epoch 1/3 | Epoch time: 774.23s | Total elapsed: 798.52s
epoch 1: val EM 66.78  F1 80.81
epoch 2/3 [RL+CE]: 100% 2646/2646 [12:52<00:00,  3.42it/s, ce=0.423, reward=0.864]
Epoch 2/3 | Epoch time: 772.58s | Total elapsed: 1581.97s
epoch 2: val EM 67.30  F1 81.27
epoch 3/3 [RL+CE]: 100% 2646/2646 [12:52<00:00,  3.42it/s, ce=0.385, reward=0.881]
Epoch 3/3 | Epoch time: 772.72s | Total elapsed: 2365.82s
epoch 3: val EM 67.56  F1 81.51
[rl] dev EM 79.45  F1 87.39


### Error Analysis

In [15]:
import json
import random
import re
import string
import urllib.request
import pandas as pd

SQUAD_DEV_URL = "https://rajpurkar.github.io/SQuAD-explorer/dataset/dev-v1.1.json"
PREDICTIONS_PATH = "/content/drive/MyDrive/MS-AI/NLP-ProgrammingAssignment/ProgrammingAssignment-2/Extractive-Quesion-Answering/runs/rl/predictions.json"
SEED = 42
NUM_SAMPLES = 10


def normalize_answer(s: str) -> str:
    """Official SQuAD v1.1 answer normalization."""
    s = s.lower()
    s = "".join(ch for ch in s if ch not in set(string.punctuation))
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    return " ".join(s.split())


def get_error_df() -> pd.DataFrame:
    with urllib.request.urlopen(SQUAD_DEV_URL) as response:
        squad_data = json.loads(response.read().decode("utf-8"))

    with open(PREDICTIONS_PATH, "r", encoding="utf-8") as f:
        predictions = json.load(f)

    incorrect = []

    for article in squad_data["data"]:
        for paragraph in article["paragraphs"]:
            for qa in paragraph["qas"]:
                qid = qa["id"]
                ground_truths = [ans["text"] for ans in qa["answers"]]
                pred = predictions.get(qid, "")

                if normalize_answer(pred) not in [normalize_answer(gt) for gt in ground_truths]:
                    incorrect.append({
                        "ID": qid,
                        "Question": qa["question"],
                        "Prediction": pred,
                        "Ground Truth": " | ".join(dict.fromkeys(ground_truths)),
                    })

    random.seed(SEED)
    samples = random.sample(incorrect, min(NUM_SAMPLES, len(incorrect)))
    print(f"Total incorrect (Exact Match): {len(incorrect)}")
    return pd.DataFrame(samples)


df = get_error_df()

# Prevent pandas from truncating long questions/answers in terminal output
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 1000)

print(df.to_string(index=False))

Total incorrect (Exact Match): 2172
                      ID                                                                                                      Question                                                                                                             Prediction                                                                                                  Ground Truth
56f826a7a6d7ea1400e17429                                                        Why is the statement doubtful in the eyes of scholars?                                                                 consider the evidence for these words to be unreliable                                                               not recorded | not recorded in witness accounts
56d9c660dc89441400fdb7ea                                     When is the last time a fumble return touchdown happened in a Super Bowl?                                                                                                      

## Model#3: Using RoBERTa-Base

In [5]:
!python squad_span_qa.py --approach roberta                            # 3. RoBERTa + MLP span heads

config.json: 100% 481/481 [00:00<00:00, 2.00MB/s]
tokenizer_config.json: 100% 25.0/25.0 [00:00<00:00, 18.0kB/s]
vocab.json: 100% 899k/899k [00:00<00:00, 1.96MB/s]
merges.txt: 100% 456k/456k [00:00<00:00, 845kB/s]
tokenizer.json: 100% 1.36M/1.36M [00:00<00:00, 34.2MB/s]

model.safetensors: downloading bytes:  35% 173M/499M [00:01<00:01, 197MB/s, 11.9MB/s  ]
model.safetensors: downloading bytes:  62% 309M/499M [00:01<00:00, 321MB/s, 23.2MB/s  ]
model.safetensors: reconstructing file:  67% 335M/499M [00:01<00:00, 201MB/s, 12.9MB/s  ]
model.safetensors: downloading bytes: 100% 335M/335M [00:02<00:00, 137MB/s, 31.2MB/s  ]
model.safetensors: reconstructing file: 100% 499M/499M [00:02<00:00, 203MB/s, 46.7MB/s  ]
Loading weights: 100% 197/197 [00:00<00:00, 5896.64it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
l

### Error Analysis

In [16]:
import json
import random
import re
import string
import urllib.request
import pandas as pd

SQUAD_DEV_URL = "https://rajpurkar.github.io/SQuAD-explorer/dataset/dev-v1.1.json"
PREDICTIONS_PATH = "/content/drive/MyDrive/MS-AI/NLP-ProgrammingAssignment/ProgrammingAssignment-2/Extractive-Quesion-Answering/runs/roberta/predictions.json"
SEED = 42
NUM_SAMPLES = 10


def normalize_answer(s: str) -> str:
    """Official SQuAD v1.1 answer normalization."""
    s = s.lower()
    s = "".join(ch for ch in s if ch not in set(string.punctuation))
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    return " ".join(s.split())


def get_error_df() -> pd.DataFrame:
    with urllib.request.urlopen(SQUAD_DEV_URL) as response:
        squad_data = json.loads(response.read().decode("utf-8"))

    with open(PREDICTIONS_PATH, "r", encoding="utf-8") as f:
        predictions = json.load(f)

    incorrect = []

    for article in squad_data["data"]:
        for paragraph in article["paragraphs"]:
            for qa in paragraph["qas"]:
                qid = qa["id"]
                ground_truths = [ans["text"] for ans in qa["answers"]]
                pred = predictions.get(qid, "")

                if normalize_answer(pred) not in [normalize_answer(gt) for gt in ground_truths]:
                    incorrect.append({
                        "ID": qid,
                        "Question": qa["question"],
                        "Prediction": pred,
                        "Ground Truth": " | ".join(dict.fromkeys(ground_truths)),
                    })

    random.seed(SEED)
    samples = random.sample(incorrect, min(NUM_SAMPLES, len(incorrect)))
    print(f"Total incorrect (Exact Match): {len(incorrect)}")
    return pd.DataFrame(samples)


df = get_error_df()

# Prevent pandas from truncating long questions/answers in terminal output
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 1000)

print(df.to_string(index=False))

Total incorrect (Exact Match): 1500
                      ID                                                                                                   Question                                                                                               Prediction                                                                          Ground Truth
572f59b4a23a5019007fc588                                                    Where does the cold and warm water meet the warm water?                                                                                         near the surface                                                               Rheinbrech | Upper Lake
56e7560937bdd419002c3e8e                                          The teacher's in a platoon style teaching are usually more _____?                                                                           knowledgeable in that one area                                                                         knowledge

## Comparing Model Performance

In [6]:
!python squad_span_qa.py --compare runs                                # EM/F1 table of all runs

approach    encoder                   dev EM  dev F1
rl          runs/token_cls/best        79.45   87.39
roberta     roberta-base               85.81   92.13
token_cls   bert-base-uncased          80.24   87.88


### Downloading The Folder

In [8]:
!zip -r /content/EQA.zip /content/Extractive-Quesion-Answering

  adding: content/Extractive-Quesion-Answering/ (stored 0%)
  adding: content/Extractive-Quesion-Answering/README.md (stored 0%)
  adding: content/Extractive-Quesion-Answering/.git/ (stored 0%)
  adding: content/Extractive-Quesion-Answering/.git/refs/ (stored 0%)
  adding: content/Extractive-Quesion-Answering/.git/refs/remotes/ (stored 0%)
  adding: content/Extractive-Quesion-Answering/.git/refs/remotes/origin/ (stored 0%)
  adding: content/Extractive-Quesion-Answering/.git/refs/remotes/origin/HEAD (stored 0%)
  adding: content/Extractive-Quesion-Answering/.git/refs/tags/ (stored 0%)
  adding: content/Extractive-Quesion-Answering/.git/refs/heads/ (stored 0%)
  adding: content/Extractive-Quesion-Answering/.git/refs/heads/main (stored 0%)
  adding: content/Extractive-Quesion-Answering/.git/info/ (stored 0%)
  adding: content/Extractive-Quesion-Answering/.git/info/exclude (deflated 28%)
  adding: content/Extractive-Quesion-Answering/.git/hooks/ (stored 0%)
  adding: content/Extractive-Que

In [9]:
from google.colab import files

# Trigger the browser download
files.download('/content/EQA.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>